In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
import nltk
from bs4 import BeautifulSoup
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
import re
from sklearn.svm import SVC  
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.metrics import  roc_curve, auc
import matplotlib.pyplot as plt
import pickle

In [2]:
#Get the dataset from the file
df = pd.read_csv('IMDB Dataset.csv')
#Test the dataste so i display the first 10 rows of the dataset
print(df.head(10))

                                              review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive
2  I thought this was a wonderful way to spend ti...  positive
3  Basically there's a family where a little boy ...  negative
4  Petter Mattei's "Love in the Time of Money" is...  positive
5  Probably my all-time favorite movie, a story o...  positive
6  I sure would like to see a resurrection of a u...  positive
7  This show was an amazing, fresh & innovative i...  negative
8  Encouraged by the positive comments about this...  negative
9  If you like original gut wrenching laughter yo...  positive


In [3]:
#Labed postive = 1, negative =0
label = LabelEncoder()
df['sentiment'] = label.fit_transform(df['sentiment'])
# Download NLTK stopwords
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\kawoo\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [4]:
df['sentiment'] 

0        1
1        1
2        1
3        0
4        1
        ..
49995    1
49996    0
49997    0
49998    0
49999    0
Name: sentiment, Length: 50000, dtype: int32

In [ ]:
#Extracting reviews and sentiments to x and y
x = df['review']
y = df['sentiment']

In [ ]:
# Initializing Porter Stemmer for stemming words
ps = PorterStemmer()
Corpus = []

# Preprocessing the review
for i in range(len(x)):
    # Remove HTML tags
    review_without_html = BeautifulSoup(x[i], 'html.parser').get_text()

    # Remove non-letter characters
    review = re.sub('[^a-zA-Z]', ' ', review_without_html)
    # make the word to lower cases
    review = review.lower()
    # Do tokenization with NLTK
    review = review.split()
    # Stemming
    review = [ps.stem(word) for word in review if word not in set(stopwords.words('english'))]
    review = ' '.join(review)
    Corpus.append(review)

In [ ]:
# Create a TF-IDF vectorizer with a maximum 5000 features
cv = TfidfVectorizer(max_features = 5000)
x = cv.fit_transform(Corpus).toarray()
x.shape

In [ ]:
# Split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)
X_train.shape, X_test.shape, y_train.shape, y_test.shape

In [ ]:
from sklearn.svm import SVC  
#Train the SVM model
svm_model = SVC()
svm_model.fit(X_train, y_train)
# Make predictions on the test set
y_pred = svm_model.predict(X_test)



In [ ]:
#Print the accuracy and metrics
print(accuracy_score(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

# Create a DataFrame to display actual and predicted sentiments
pd.DataFrame(np.c_[y_test, y_pred], columns=['Actual', 'Predicted'])

In [ ]:
# Save the vectorizer and the trained model with pickle
pickle.dump(cv, open('count-Vectorizer.pkl', 'wb'))
pickle.dump(svm_model, open('Movies_Review_Classification.pkl', 'wb'))
# Load the saved vectorizer and model
save_vectorizer = pickle.load(open('count-Vectorizer.pkl', 'rb'))
model = pickle.load(open('Movies_Review_Classification.pkl', 'rb'))
# The function to test the model with new review
def test_model(sentence):
    sen = save_vectorizer.transform([sentence]).toarray()
    res = model.predict(sen)[0]
    if res == 1:
        return 'Positive review'
    else:
        return 'negative review'
sen = 'This is the good movie'
res = test_model(sen)
print(res)

In [ ]:
# ROC Analysis
fpr, tpr, thresholds = roc_curve(y_test, y_pred)
roc_auc = auc(fpr, tpr)

In [ ]:
# Plot ROC curve
plt.figure()
plt.plot(fpr, tpr, color='darkorange', lw=2, label='ROC curve (area = %0.2f)' % roc_auc)
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic')
plt.legend(loc="lower right")
plt.show()